# Adversarial Self-Verification: Cutting False Positives in Agent Output

Single-pass LLM agents produce plausible-sounding but factually incorrect outputs surprisingly often — estimates range from 15–30% on knowledge-intensive tasks. The usual fix — retry loops — does not help because the same model, given the same prompt, tends to reproduce the same errors.

This notebook demonstrates **adversarial self-verification**: instead of asking a second model *"is this correct?"* (which triggers sycophantic agreement), we prompt independent verifier agents to *actively try to find flaws*. The key insight is that the framing of the verification prompt determines whether you get genuine critique or rubber-stamp approval.

## What you will build

```
User question
      │
      ▼
┌─────────────┐
│  Generator  │  claude-sonnet-4-6 answers the question
└──────┬──────┘
       │ initial answer
       ▼
┌──────────────────────────────┐
│  3 Parallel Verifier Agents  │  Each is told: "Find what is WRONG"
│  (claude-haiku-4-5)          │  asyncio.gather — all run at once
└──────┬───────────────────────┘
       │ 3 critiques
       ▼
┌─────────────┐
│  Arbitrator │  Count real issues; if ≥ 2/3 verifiers flag problems
└──────┬──────┘  → regenerate with critique embedded
       │
       ▼
  Verified answer
```

## Why adversarial prompting?

LLMs are trained on human feedback, and humans tend to prefer agreeable responses. When you ask *"is this correct?"*, the model defaults to finding reasons it is correct. When you ask *"what is wrong with this?"*, it is forced to reason in a different mode — one that surfaces genuine issues it would otherwise suppress.

This is not a hack. It is the same cognitive strategy humans use in red-teaming, code review, and adversarial collaboration in science.

## Setup

Install the Anthropic SDK and configure your API key.

In [ ]:
%pip install anthropic python-dotenv --quiet

In [ ]:
import asyncio
import os
import time
from dataclasses import dataclass
from textwrap import dedent

import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.AsyncAnthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

GENERATOR_MODEL = "claude-sonnet-4-6"   # High-quality generation
VERIFIER_MODEL = "claude-haiku-4-5"     # Fast, cheap, independent verifiers

print(f"Generator : {GENERATOR_MODEL}")
print(f"Verifiers : {VERIFIER_MODEL} (3 parallel instances)")

## Stage 1 — The Generator

The generator is a straightforward single-pass Claude call. No special prompting. We deliberately do not ask it to double-check itself — that is the verifiers' job.

In [ ]:
GENERATOR_SYSTEM = dedent("""
    You are a knowledgeable technical assistant. Answer the user's question
    accurately and in detail. Cite specific version numbers, benchmarks, or
    mechanisms when relevant. Do not hedge excessively — give a direct answer.
""").strip()


async def generate_answer(question: str) -> tuple[str, float]:
    """Run the generator and return (answer_text, elapsed_seconds)."""
    t0 = time.perf_counter()
    response = await client.messages.create(
        model=GENERATOR_MODEL,
        max_tokens=1024,
        system=GENERATOR_SYSTEM,
        messages=[{"role": "user", "content": question}],
    )
    elapsed = time.perf_counter() - t0
    text = response.content[0].text
    return text, elapsed

## Stage 2 — The Adversarial Verifiers

This is where the pattern differs from naive self-correction.

**Wrong approach** (sycophantic):
> "Please review this answer and tell me if it is correct."

**Right approach** (adversarial):
> "Your job is to find what is WRONG with this answer. Assume it contains at least one error. Find it."

Each verifier receives a randomly-rotated system prompt variant to reduce correlation between their outputs. Three independent agents running in parallel gives us a majority-vote signal.

In [ ]:
# Three prompt variants — different framings of the same adversarial goal.
# Rotating prompts reduces echo-chamber effects where all verifiers notice
# the same surface-level issue and miss deeper ones.
VERIFIER_PROMPTS = [
    dedent("""
        You are a rigorous fact-checker. Your ONLY job is to find errors.
        You will be given a technical answer. Assume it contains at least one
        factual error, outdated claim, or logical gap.

        Identify the SINGLE most serious problem. Be specific: quote the
        problematic sentence, explain why it is wrong, and what the correct
        information is.

        If you genuinely cannot find an error after careful review, say
        "NO_ISSUE_FOUND" and briefly explain why you are confident.

        Do NOT validate, praise, or agree with anything. Your role is critique only.
    """).strip(),

    dedent("""
        You are a skeptical expert reviewer. You distrust LLM-generated technical
        content by default. You will be given an answer to a technical question.

        Hunt for: incorrect version numbers, reversed causality, missing caveats
        about edge cases, or claims that sound plausible but are subtly wrong.

        Report the most important flaw you find. Quote the exact text that is
        wrong and provide the correction.

        If you are confident the answer is accurate, respond with "NO_ISSUE_FOUND"
        and explain what you checked.
    """).strip(),

    dedent("""
        You are playing devil's advocate against the following technical answer.
        Your goal: find a claim that a domain expert would push back on.

        Focus on: unsupported absolutes ("always", "never", "the only way"),
        claims that changed between major version releases, and performance
        numbers cited without context.

        State exactly what is wrong and what a corrected version would say.

        If you cannot identify a genuine problem, respond with "NO_ISSUE_FOUND"
        and what you examined.
    """).strip(),
]


@dataclass
class VerifierResult:
    verifier_id: int
    critique: str
    found_issue: bool  # True if the verifier flagged a real problem
    elapsed: float


async def run_verifier(verifier_id: int, question: str, answer: str) -> VerifierResult:
    """Run one adversarial verifier agent."""
    system_prompt = VERIFIER_PROMPTS[verifier_id % len(VERIFIER_PROMPTS)]

    user_message = dedent(f"""
        ORIGINAL QUESTION:
        {question}

        ANSWER TO CRITIQUE:
        {answer}

        Find what is wrong with this answer.
    """).strip()

    t0 = time.perf_counter()
    response = await client.messages.create(
        model=VERIFIER_MODEL,
        max_tokens=512,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    elapsed = time.perf_counter() - t0

    critique = response.content[0].text
    found_issue = "NO_ISSUE_FOUND" not in critique

    return VerifierResult(
        verifier_id=verifier_id,
        critique=critique,
        found_issue=found_issue,
        elapsed=elapsed,
    )


async def run_parallel_verifiers(
    question: str, answer: str, n: int = 3
) -> list[VerifierResult]:
    """Run N verifiers in parallel using asyncio.gather."""
    tasks = [run_verifier(i, question, answer) for i in range(n)]
    return await asyncio.gather(*tasks)

## Stage 3 — Synthesis and Conditional Regeneration

The arbitrator counts how many verifiers flagged real issues. If 2 or more out of 3 did, we treat the original answer as unreliable and regenerate — this time embedding the critiques directly into the generator prompt.

**Majority vote (2/3) threshold** is intentional:
- 1/3 flagging an issue might be one verifier being overly aggressive
- 2/3 flagging issues is a strong signal that there is a genuine problem
- 3/3 would be too conservative — verifiers sometimes disagree on severity

In [ ]:
REGENERATOR_SYSTEM = dedent("""
    You are a technical expert. A previous answer to the user's question was
    reviewed by independent fact-checkers who found problems. You will be given
    the original answer AND the critiques.

    Your task: write a corrected, improved answer that directly addresses each
    critique. Where a critique is wrong or overly pedantic, you may note that
    briefly and explain why the original was correct. Be precise and accurate.
""").strip()


async def regenerate_with_critique(
    question: str,
    original_answer: str,
    critiques: list[VerifierResult],
) -> tuple[str, float]:
    """Regenerate the answer with critique embedded in the prompt."""
    critique_block = "\n\n".join(
        f"CRITIQUE {r.verifier_id + 1}:\n{r.critique}"
        for r in critiques
        if r.found_issue
    )

    user_message = dedent(f"""
        QUESTION: {question}

        ORIGINAL ANSWER (may contain errors):
        {original_answer}

        CRITIQUES FROM INDEPENDENT REVIEWERS:
        {critique_block}

        Please write a corrected answer that fixes the issues raised above.
    """).strip()

    t0 = time.perf_counter()
    response = await client.messages.create(
        model=GENERATOR_MODEL,
        max_tokens=1024,
        system=REGENERATOR_SYSTEM,
        messages=[{"role": "user", "content": user_message}],
    )
    elapsed = time.perf_counter() - t0
    return response.content[0].text, elapsed


def print_separator(label: str = "", width: int = 70) -> None:
    if label:
        pad = (width - len(label) - 2) // 2
        print("=" * pad + f" {label} " + "=" * pad)
    else:
        print("=" * width)

## The Full Pipeline

Assemble all three stages into one `verify_and_correct` function.

In [ ]:
async def verify_and_correct(question: str, issue_threshold: int = 2) -> dict:
    """
    Full adversarial self-verification pipeline.

    Args:
        question: The question to answer and verify.
        issue_threshold: Number of verifiers that must flag issues
                         before triggering regeneration (default 2/3).

    Returns:
        dict with keys: question, initial_answer, verifier_results,
                        regenerated, final_answer, total_elapsed.
    """
    pipeline_start = time.perf_counter()

    # ── Stage 1: Generate ──────────────────────────────────────────────────
    print_separator("STAGE 1: GENERATION")
    print(f"Question: {question}\n")
    initial_answer, gen_time = await generate_answer(question)
    print(f"[Generator completed in {gen_time:.2f}s]\n")
    print(initial_answer)

    # ── Stage 2: Adversarial verification (parallel) ───────────────────────
    print_separator("STAGE 2: ADVERSARIAL VERIFICATION (3 parallel agents)")
    print("Spawning 3 independent adversarial verifiers...\n")

    verify_start = time.perf_counter()
    verifier_results = await run_parallel_verifiers(question, initial_answer, n=3)
    verify_elapsed = time.perf_counter() - verify_start

    issues_found = sum(1 for r in verifier_results if r.found_issue)

    for r in verifier_results:
        flag = "ISSUE FOUND" if r.found_issue else "NO ISSUE"
        print(f"--- Verifier {r.verifier_id + 1} [{flag}] ({r.elapsed:.2f}s) ---")
        print(r.critique)
        print()

    print(f"[All 3 verifiers completed in {verify_elapsed:.2f}s (parallel)]")
    print(f"Issues flagged: {issues_found}/3  |  Threshold: {issue_threshold}/3")

    # ── Stage 3: Synthesize ────────────────────────────────────────────────
    print_separator("STAGE 3: SYNTHESIS")
    regenerated = issues_found >= issue_threshold

    if regenerated:
        print(
            f"Majority vote: {issues_found}/3 verifiers found issues. "
            "Regenerating with critique embedded...\n"
        )
        final_answer, regen_time = await regenerate_with_critique(
            question, initial_answer, verifier_results
        )
        print(f"[Regeneration completed in {regen_time:.2f}s]\n")
    else:
        print(
            f"Minority flag: only {issues_found}/3 verifiers found issues. "
            "Original answer passes verification.\n"
        )
        final_answer = initial_answer

    print_separator("FINAL VERIFIED ANSWER")
    print(final_answer)

    total_elapsed = time.perf_counter() - pipeline_start
    print_separator()
    print(f"Total pipeline time: {total_elapsed:.2f}s  |  Regenerated: {regenerated}")
    print_separator()

    return {
        "question": question,
        "initial_answer": initial_answer,
        "verifier_results": verifier_results,
        "issues_found": issues_found,
        "regenerated": regenerated,
        "final_answer": final_answer,
        "total_elapsed": total_elapsed,
    }

## Concrete Example: Python GIL in CPython 3.12

We choose this question deliberately. CPython 3.12 introduced **PEP 703** ("Making the Global Interpreter Lock Optional"), which changed some long-held assumptions about the GIL. An LLM trained before or around that release might state outdated facts about GIL behavior — exactly the kind of subtle error that adversarial verifiers are designed to catch.

This is a good test case because:
- The facts are checkable (not opinion-based)
- Recent version-specific changes make outdated answers plausible-sounding
- Domain experts would spot version-specific errors immediately

In [ ]:
QUESTION = (
    "What are the performance characteristics of Python's GIL in CPython 3.12? "
    "Specifically: how does it affect CPU-bound vs I/O-bound multithreading, "
    "what changed in 3.12 compared to 3.11, and what does PEP 703 change "
    "for the GIL's future?"
)

# Run the full pipeline
result = await verify_and_correct(QUESTION)

## Interpreting the Results

Print a clean summary of what happened at each stage.

In [ ]:
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"Verifiers that flagged issues : {result['issues_found']}/3")
print(f"Regeneration triggered        : {result['regenerated']}")
print(f"Total wall-clock time         : {result['total_elapsed']:.1f}s")
print()
print("VERIFIER VERDICTS")
for r in result["verifier_results"]:
    status = "FLAGGED" if r.found_issue else "PASSED"
    print(f"  Verifier {r.verifier_id + 1}: {status} ({r.elapsed:.2f}s)")
print()
if result["regenerated"]:
    print("Outcome: Answer was improved using embedded critique.")
else:
    print("Outcome: Original answer passed adversarial review.")

## Anti-Sycophancy: Why Prompt Framing is Everything

The most common mistake when building verification pipelines is asking the verifier the wrong question. Here is a direct comparison:

In [ ]:
# A deliberately brief, slightly wrong answer to test both prompting strategies.
TEST_ANSWER = dedent("""
    Python's GIL prevents true parallelism for CPU-bound tasks. In CPython 3.12,
    the GIL was completely removed, allowing true multi-core parallelism for all
    Python programs by default. This was the main change introduced in 3.12.
    I/O-bound tasks have always been unaffected since threads release the GIL
    during blocking I/O operations.
""").strip()

TEST_QUESTION = "What changed about the GIL in CPython 3.12?"


async def compare_verification_prompts(question: str, answer: str) -> None:
    """Run the same answer through sycophantic vs adversarial verification."""

    sycophantic_prompt = (
        "Please review the following answer and let me know if it is correct."
    )
    adversarial_prompt = (
        "Find what is WRONG with the following answer. Assume it contains at "
        "least one factual error. Identify it specifically and explain the correction."
    )

    async def run_check(label: str, instruction: str) -> None:
        message = await client.messages.create(
            model=VERIFIER_MODEL,
            max_tokens=300,
            messages=[
                {
                    "role": "user",
                    "content": (
                        f"{instruction}\n\n"
                        f"QUESTION: {question}\n\n"
                        f"ANSWER: {answer}"
                    ),
                }
            ],
        )
        print_separator(label)
        print(message.content[0].text)
        print()

    print_separator("ANTI-SYCOPHANCY DEMONSTRATION")
    print(f"Test answer (contains a factual error):\n{answer}\n")
    await run_check("SYCOPHANTIC PROMPT (wrong approach)", sycophantic_prompt)
    await run_check("ADVERSARIAL PROMPT (right approach)", adversarial_prompt)


await compare_verification_prompts(TEST_QUESTION, TEST_ANSWER)

**What you should observe above:** The sycophantic prompt tends to surface minor caveats while broadly agreeing with the answer. The adversarial prompt homes in on the specific factual error: PEP 703 does not remove the GIL by default in 3.12 — it made the GIL *optional* and experimental, with `PYTHON_GIL=0` as an opt-in mechanism. Removing the GIL by default is targeted for a later release.

This is the core mechanism of adversarial self-verification: **the prompt framing, not a different model, is what unlocks genuine critique**.

## When to Apply This Pattern

Adversarial self-verification adds latency and cost. Apply it selectively:

| Task type | Use adversarial verification? |
|-----------|-------------------------------|
| Factual Q&A about recent/versioned topics | Yes |
| Technical documentation generation | Yes |
| Medical, legal, financial summaries | Yes |
| Creative writing | No |
| Simple classification | No |
| Code generation (unit tests cover this) | Situational |

## Cost and Latency Profile

Using `claude-haiku-4-5` for verifiers (3 parallel) keeps cost manageable:

- **Without regeneration**: 1× Sonnet call + 3× Haiku calls (parallel)
- **With regeneration**: 2× Sonnet calls + 3× Haiku calls (parallel)
- **Parallel verifiers** run in ~the same wall-clock time as 1 verifier
- **Haiku** is roughly 20–25× cheaper per token than Sonnet

The 3 parallel Haiku verifiers cost roughly the same as 10-15% of a single Sonnet call — a cheap insurance policy against a 15–30% error rate.

## Extensions

1. **Confidence scoring**: Have each verifier assign a 0–10 confidence score to their critique. Weight the majority vote by confidence.
2. **Domain-specific verifier personas**: "You are a CPython core developer reviewing this answer" vs "You are a CS professor."
3. **Structured critique output**: Use tool use to force verifiers to output `{claim, error_type, severity, correction}` JSON — easier to parse for downstream systems.
4. **Iterative refinement**: Run verification again on the regenerated answer (limit to 2 iterations to avoid infinite loops).
5. **Citation injection**: After verification, a final agent adds citations for every factual claim in the answer.

In [ ]:
# Run on a second question to see the pipeline on a different domain
QUESTION_2 = (
    "What is the difference between a transformer's self-attention and "
    "cross-attention mechanisms, and in which components of a standard "
    "encoder-decoder architecture does each appear?"
)

result2 = await verify_and_correct(QUESTION_2)

## Summary

Adversarial self-verification is a lightweight, deployable pattern for reducing false positives in agent output:

1. **Generate** normally with the best available model (Sonnet).
2. **Verify** with 3 parallel adversarial agents (Haiku) — each prompted to *find problems*, not validate.
3. **Synthesize** using majority vote: if 2/3 flag issues, regenerate with critique embedded.

The critical insight: **prompt framing is the mechanism**. Asking "is this correct?" produces agreement. Asking "what is wrong?" produces genuine critique. This is not a model capability gap — it is a prompting gap. The same model, given an adversarial prompt, surfaces errors it would otherwise suppress in the name of being helpful.

**Key code patterns used:**
- `asyncio.gather` for parallel verifier calls (cuts wall-clock time by 2–3×)
- `anthropic.AsyncAnthropic` for async API calls
- Rotating system prompt variants across verifiers to reduce correlation
- `NO_ISSUE_FOUND` sentinel for clean binary classification of verifier output
- Majority vote threshold (2/3) before triggering expensive regeneration